Install Required Libraries


In [1]:
# Only math & vector ops needed
!pip install numpy


## NumPy is used for normalization, fusion, and cosine similarity

In [2]:
import numpy as np


Input Vectors (Temporary Test Inputs)
These simulate outputs from M1 and the appearance team.
Later, replace them with real outputs


In [3]:
# Simulated gait vector from M1
V_gait = np.random.randn(512).astype(np.float32)

# Simulated appearance vector from Team A
V_app = np.random.randn(512).astype(np.float32)

print(V_gait.shape, V_app.shape)


(512,) (512,)


These metrics are produced by an upstream perception/quality module,
NOT by M1 or M2

In [4]:
# Quality metrics coming from packet
quality_metrics = {
    "blur_score": 0.2,        # 0 = sharp, 1 = very blurry
    "avg_height": 170.0,      # in cm
    "occlusion_flag": False   # True if occluded
}


Reliability Weight (Alpha Calculation)

In [18]:
# This function converts quality metrics into a reliability weight (alpha)
# alpha ∈ [0, 1]
# Higher alpha → trust appearance more
# Lower alpha → trust gait more

def calculate_alpha(quality_metrics):
    """
    Computes reliability weight alpha ∈ [0, 1]
    Higher alpha → trust appearance more
    """

    blur = quality_metrics["blur_score"]        # [0,1]
    occlusion = quality_metrics["occlusion_flag"]

    # Base trust on gait
    gait_trust = 1.0 - blur

    # Penalize occlusion
    if occlusion:
        gait_trust *= 0.5

    # Clamp to valid range
    gait_trust = np.clip(gait_trust, 0.0, 1.0)

    # Alpha controls appearance weight
    alpha = 1.0 - gait_trust

    return alpha


Compute Alpha
This alpha value controls fusion weighting


In [6]:
alpha = calculate_alpha(quality_metrics)
print("Alpha:", alpha)


Alpha: 0.19999999999999996


Vector Normalization

In [19]:
# L2 normalization ensures vector magnitude = 1

def l2_normalize(vec):
    """
    L2 normalization of a vector
    """
    norm = np.linalg.norm(vec)
    if norm == 0:
        return vec
    return vec / norm


Normalize Input Vectors

In [20]:
V_gait_norm = l2_normalize(V_gait)
V_app_norm = l2_normalize(V_app)


print("Gait norm:", np.linalg.norm(V_gait_norm))
print("App norm:", np.linalg.norm(V_app_norm))



Gait norm: 1.0
App norm: 1.0


Feature Fusion Logic (Combines gait and appearance into a single identity vector)

In [21]:
# Weighted fusion of appearance and gait vectors
def fuse_vectors(app_vec, gait_vec, alpha):
    fused_vector = alpha * app_vec + (1 - alpha) * gait_vec
    return l2_normalize(fused_vector)


Fuse Vectors(Produces the final identity vector)

In [22]:
V_final = fuse_vectors(V_app_norm, V_gait_norm, alpha)
print("Final vector shape:", V_final.shape)


Final vector shape: (512,)


Gallery (Identity Database)

(Simple in-memory structure for storing known identities)

In [23]:
# Dictionary to store identity vectors
gallery = {}


Add New Identity

(Used during enrollment phase)

In [13]:
def add_identity(person_id, vector):
    """
    Add new identity to gallery
    """
    gallery[person_id] = vector


Update Existing Identity

(Uses exponential moving average to adapt over time)

In [24]:
def update_identity(person_id, vector, momentum=0.7):
    if person_id not in gallery:
        gallery[person_id] = vector
    else:
        gallery[person_id] = l2_normalize(
            momentum * gallery[person_id] +
            (1 - momentum) * vector
        )


Add a Test Identity

(Simulates adding a known person to the gallery)

In [25]:
add_identity("Person_1", V_final)
print("Gallery contents:", gallery.keys())


Gallery contents: dict_keys(['Person_1'])


Cosine Similarity Function

(Measures similarity between two identity vectors)

In [26]:
def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2)


Identity Matching Logic

(Compares final vector against gallery to identify person)

In [27]:
def identify(final_vector, threshold=0.75):
    best_id = None
    best_score = -1.0

    for person_id, stored_vector in gallery.items():
        score = cosine_similarity(final_vector, stored_vector)

        if score > best_score:
            best_score = score
            best_id = person_id

    if best_score >= threshold:
        return best_id, best_score
    else:
        return "Unknown", best_score


Run Identification

In [28]:
result = identify(V_final)
print("Identification result:", result)



Identification result: ('Person_1', np.float64(1.0000000000000002))
